In [1]:
from pathlib import Path
import sys
import scanpy as sc
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import session_info
import warnings
warnings.filterwarnings('ignore')


# Decrease this for testing, default 250
MAX_NEPOCHS = 250

/home/workspace/environment/sc-persist/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Import the reference datasets

* GSE139088 
* GSE254789 <- current script

In [2]:
base_dir = Path('/home/workspace/private/projects/kim/drg')
panel_dir = base_dir / 'panel/gene-lists'

adata_path = base_dir / 'data/scrna-seq/h5ad/04_clustered/GSE254789-nonneurons.h5ad'
bdata_path =

kim_path = panel_dir / 'input/zhen-gene-list.csv'
flow_path = panel_dir / 'input/flow-panel-genes.csv'

output_dir = panel_dir / 'output'
Path(output_dir).mkdir(parents=True, exist_ok=True)

In [5]:
adata = sc.read_h5ad(input_dir / "GSE139088-scvi-leiden.h5ad")
bdata = sc.read_h5ad(input_dir / "GSE254789-nonneurons.h5ad")

## Manual gene selection

These are the manually selected genes, that will be included in the final gene panel. PERSIST will choose the best genes to complement this manual selection.

### Helper functions

In [4]:
def all_genes_in_panel(panel: dict) -> list:
    """
    Returns a set of unique genes from the given panel.
    
    Args:
        panel (dict): A dictionary representing a genomic panel, where keys are sample names and values are lists of gene names for each sample.
    
    Returns:
        set: A set of unique genes found in the given panel.
    """
    return list(set(sum(panel.values(), [])))

def print_panel(panel: dict):
    """
    Prints information about the given panel, including the number of unique genes and a list of genes for each sample.
    
    Args:
        panel (dict): A dictionary representing a genomic panel, where keys are sample names and values are lists of gene names for each sample.
    
    Returns:
        None
    """
    class color:
        PURPLE = '\033[95m'
        CYAN = '\033[96m'
        DARKCYAN = '\033[36m'
        BLUE = '\033[94m'
        GREEN = '\033[92m'
        YELLOW = '\033[93m'
        RED = '\033[91m'
        BOLD = '\033[1m'
        UNDERLINE = '\033[4m'
        END = '\033[0m'

    print(color.UNDERLINE + color.BOLD + "Current Panel" + color.END)
    print(f"currently {color.PURPLE}{len(all_genes_in_panel(panel))}{color.END} unique genes")
    for k in panel.keys():
        import textwrap
        genes = ', '.join(panel[k])
        genes = '\n'.join(textwrap.wrap(genes, 128))
        print(color.BOLD + k + color.END + f" ({len(panel[k])} genes): " + genes)


### Gene lists

In [5]:
# Kim lab gene list
kim_list = pd.read_csv(kim_path)
kim_list = kim_list['gene'].unique()
kim_list = list(kim_list)
print("Kim list length:", len(kim_list), "genes")

# Spectral flow panel genes
flow_list = pd.read_csv(flow_path)
flow_list = flow_list['gene'].unique()
flow_list = list(flow_list)
print("Spectral flow panel list length:", len(flow_list), "genes")

Kim list length: 115 genes
Spectral flow panel list length: 51 genes


### Putting it together

In [6]:
panel = {
    "zhen_list": kim_list,
    "spectral_panel": flow_list,
    "GSE254789_supervised": [],
    "GSE254789_unsupervised": [],
}
print_panel(panel)

Current Panel
currently 165 unique genes
zhen_list (115 genes): Nppb, Sst, Cysltr1, Cysltr2, Il31ra, Il4ra, Il13ra1, Il13ra2, Osmr, Jak1, Stat6, Piezo2, Adra2a, Bmpr1b, Nefh, Th, Ret, Cd274,
Pdcd1, Kit, Ntrk1, Ntrk2, Ntrk3, Scn8a, Scn9a, Scn10a, Scn11a, Tac1, Calca, Calcb, Nmb, Nts, Adcyap1, Mrgprd, Mrgpra1, Mrgpra3,
Mrgprc11, Mrgprb4, Trpa1, Trpv1, Trpv2, Trpv4, Trpm3, Trpm5, Trpm8, Oprk1, Oprd1, Oprm1, Asic1, Asic2, Asic3, Asic4, P2ry1,
P2ry2, P2ry4, P2rx2, P2rx3, P2rx4, Gfra1, Gfra2, Gfra3, Sstr1, Sstr2, Sstr3, Sstr4, Sstr5, Hrh1, Hrh2, Hrh3, Hrh4, Htr1a, Htr1b,
Htr1d, Htr1f, Htr2a, Htr2b, Htr2c, Htr3a, Htr3b, Htr4, Htr5a, Htr5b, Htr5c, Htr6, Htr7, F2r, F2rl1, F2rl2, F2rl3, S1pr1, S1pr2,
S1pr3, S1pr4, S1pr5, Ptger1, Ptger2, Ptger3, Ptger4, Tnfrsf1a, Il1r1, Il6st, Il17ra, Il10rb, Tlr4, Tlr5, Tlr7, Crlf2, Cbln2,
Ldhb, Necab2, Fam19a1, Smr2, Hpse, Spp1, Ntm
spectral_panel (51 genes): Ptprc, Cd8a, Cd44, Thy1, Cd4, Pdcd1, Cd8b1, Ifng, Havcr2, Gzmb, Tnf, Il17a, Prf1, Il2, H2-D1, Foxp3, Ro

In [7]:
def are_genes_in_adata(adata, panel: dict):
    """
    Checks if all genes in a panel are present in anndata.

    Parameters:
        adata (anndata.AnnData): The annotated data.
        panel (list or str): The list of gene names or a single string representing the gene name.

    Returns:
        None
    """

    # Let's see if all genes are in the anndata, and if not which are missing
    genes = all_genes_in_panel(panel)
    for gene in genes:
        if gene not in adata.var_names.values:
            print(gene, "not found")

print("drg")
are_genes_in_adata(adata, panel)


drg
Mrgprc11 not found
Htr5c not found
